In [ ]:
# Llama-3.2 is gated. Store a Kaggle secret named HF_TOKEN.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:

import os
import csv
import gc
import numpy as np
import torch
import matplotlib.pyplot as plt

# Folder for the Question 4 results
OUTPUT_DIR = "/kaggle/working/q4_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LAYERS = [0, 7, 15]
MAX_PLOT_VALUES = 200
ENERGY_THRESHOLD = 50.0

results = []

for k in LAYERS:
    print(f"\nAnalyzing Layer {k}...")

    # 1. Extract the original q_proj weight matrix
    W = model.model.layers[k].self_attn.q_proj.weight
    W = W.detach().to(device="cpu", dtype=torch.float32)

    print(f"Weight matrix shape: {tuple(W.shape)}")

    # 2. Compute singular values using SVD
    # We only need S, so svdvals is sufficient.
    S = torch.linalg.svdvals(W)

    # 3. Calculate cumulative energy using ALL singular values
    squared = S ** 2
    energy = torch.cumsum(squared, dim=0) / squared.sum() * 100

    # 4. Find the first rank that reaches at least 50% energy
    rank_50 = int(torch.searchsorted(energy, ENERGY_THRESHOLD).item()) + 1

    print(f"Total singular values: {len(S)}")
    print(f"Minimum rank for 50% energy: {rank_50}")
    print(f"Energy at rank 200: {energy[199].item():.2f}%")

    results.append({
        "layer": k,
        "rank_50": rank_50,
        "energy_at_200": energy[199].item()
    })

    # Convert to NumPy for plotting
    singular_values = S.cpu().numpy()
    cumulative_energy = energy.cpu().numpy()

    # Plot ONLY the first 200 values
    n = min(MAX_PLOT_VALUES, len(S))
    ranks = np.arange(1, n + 1)

    # 5. Singular value magnitude bar chart
    fig, ax = plt.subplots(figsize=(6, 3.5))

    ax.bar(ranks, singular_values[:n], width=1.0)

    ax.set_title(f"Layer {k}: Singular Value Magnitudes")
    ax.set_xlabel("Singular Value Index")
    ax.set_ylabel("Singular Value Magnitude")
    ax.set_xlim(0, n + 1)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        f"{OUTPUT_DIR}/q4_layer{k}_sv.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

    # 6. Cumulative energy line chart
    fig, ax = plt.subplots(figsize=(6, 3.5))

    ax.plot(ranks, cumulative_energy[:n], linewidth=2)

    # Show the 50% energy threshold
    ax.axhline(
        50,
        color="red",
        linestyle="--",
        label="50% energy"
    )

    # Mark the minimum rank if it is within the plotted range
    if rank_50 <= n:
        ax.axvline(
            rank_50,
            color="green",
            linestyle="--",
            label=f"Rank = {rank_50}"
        )

    ax.set_title(f"Layer {k}: Cumulative Energy")
    ax.set_xlabel("Target Rank")
    ax.set_ylabel("Cumulative Energy (%)")
    ax.set_xlim(1, n)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        f"{OUTPUT_DIR}/q4_layer{k}_energy.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

    # 7. Save numerical data for later analysis
    with open(
        f"{OUTPUT_DIR}/q4_layer{k}_values.csv",
        "w",
        newline=""
    ) as f:
        writer = csv.writer(f)
        writer.writerow([
            "rank",
            "singular_value",
            "cumulative_energy_percent"
        ])

        for i in range(len(S)):
            writer.writerow([
                i + 1,
                float(singular_values[i]),
                float(cumulative_energy[i])
            ])

    del W, S, squared, energy
    gc.collect()

# 8. Save summary table
with open(f"{OUTPUT_DIR}/q4_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["layer", "rank_50", "energy_at_200"]
    )
    writer.writeheader()
    writer.writerows(results)

print("\n=== QUESTION 4 RESULTS ===")
for result in results:
    print(
        f"Layer {result['layer']}: "
        f"minimum rank for 50% energy = {result['rank_50']}"
    )

print(f"\nAll results saved in: {OUTPUT_DIR}")


In [ ]:

import torch

print("Model architecture:")
print("Number of transformer layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")

print("\nQuery projection statistics:")

for layer in [0, 7, 15]:
    W = model.model.layers[layer].self_attn.q_proj.weight
    W = W.detach().float().cpu()

    S = torch.linalg.svdvals(W)
    energy = torch.cumsum(S.square(), dim=0) / S.square().sum()

    rank_50 = torch.searchsorted(energy, 0.50).item() + 1
    rank_90 = torch.searchsorted(energy, 0.90).item() + 1
    rank_95 = torch.searchsorted(energy, 0.95).item() + 1

    print(f"\nLayer {layer}")
    print(f"  Matrix shape: {tuple(W.shape)}")
    print(f"  Parameters: {W.numel():,}")
    print(f"  Largest singular value: {S[0].item():.4f}")
    print(f"  Smallest singular value: {S[-1].item():.4f}")
    print(f"  Rank for 50% energy: {rank_50}")
    print(f"  Rank for 90% energy: {rank_90}")
    print(f"  Rank for 95% energy: {rank_95}")
    print(f"  Energy at rank 16: {energy[15].item()*100:.2f}%")
    print(f"  Energy at rank 64: {energy[63].item()*100:.2f}%")
    print(f"  Energy at rank 256: {energy[255].item()*100:.2f}%")
